# A short introduction to containerized software

After spending time using nf-core pipelines to answer bioinformatic questions, we will focus on the processes that lie behind these pipelines now.

Today, we will focus on containerization, namely via Docker. 



1. Check if Docker is installed.

In [ ]:
!docker info

### What is a container?

standardized unit of software that packages an application together with everything it needs to run (code, runtime, system tools, libraries, settings)

https://www.docker.com/resources/what-container/

---

### Why do we use containers?

- reproducible software environments
- isolation of dependencies
- portability between systems
- lightweight compared with virtual machines
- fewer dependency/version conflicts

https://www.docker.com/resources/what-container/

---

### What is a docker image?

packaged, executable software; becomes a container when its running

https://www.docker.com/resources/what-container/

---

### Let's run our first docker image:

### Login to docker

In [ ]:
# This you need to do on the command line directly
!docker login

---

### Run your first docker container

In [3]:
!docker run hello-world


Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (arm64v8)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and more with a free Docker ID:
 https://hub.docker.com/

For more examples and ideas, visit:
 https://docs.docker.com/get-started/



---

### Find the container ID

In [4]:
!docker ps -a

CONTAINER ID   IMAGE         COMMAND    CREATED         STATUS                     PORTS     NAMES
43717295d2ff   hello-world   "/hello"   5 seconds ago   Exited (0) 4 seconds ago             stupefied_shannon


---

### Delete the container again, give prove its deleted

In [5]:
!docker rm 43717295d2ff

43717295d2ff


In [6]:
!docker ps -a

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


---

### FASTQC is a very useful tool as you've learned last week. Let's try and run it from command line

Link to the software: https://www.bioinformatics.babraham.ac.uk/projects/fastqc/

Please describe the steps you took to download and run the software for the example fastq file from last week below:

1. install FastQC with Conda `conda install -c bioconda fastqc`
2. confirme the installed version `fastqc --version` (FastQC v0.12.1)
3. locate the FASTQ files `find day_02/fetchngs_out/fastq -type f -name "*.fastq.gz"`
4. run FastQC on all 4 FASTQ files and saved the results
    ```bash
    fastqc \
    day_02/fetchngs_out/fastq/*.fastq.gz \
    -o day_04/fastqc_results/Conda
    ```
5. check (`ls -lh day_04/fastqc_results`) and open (`open day_04/fastqc_results/Conda/*_fastqc.html`) the generated reports 

---

### Very well, now let's try to make use of its docker container

1. create a container holding fastqc using seqera containers (https://seqera.io/containers/)
2. use the container to generate a fastqc html of the example fastq file

Creating the FastQC container
1. open [Seqera Containers](https://seqera.io/containers/)
2. search `fastqc` and select `bioconda::fastqc` (version 0.12.1)
3. create container and copy URI
4. get container with `docker pull <CONTAINER_URI>`
5. run container `docker run --rm  <CONTAINER_URI>`

In [7]:
# pull the container
!docker pull community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae

0.12.1--9971ea336a9eddae: Pulling from library/fastqc
Digest: sha256:f97f7658a191bebe2f2a007cab5b815d380f89a948d6eb1481b046ab7618bebc
Status: Image is up to date for community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae
community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae


In [ ]:
# run the container and save the results to a new "fastqc_results" directory
!docker run --rm \
  --platform linux/amd64 \
  -v "$PWD/../day_02/fetchngs_out/fastq:/data:ro" \
  -v "$PWD/fastqc_results/Docker:/results" \
  community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae \
  fastqc \
  /data/SRX19144486_SRR23195516_1.fastq.gz \
  /data/SRX19144486_SRR23195516_2.fastq.gz \
  /data/SRX19144488_SRR23195511_1.fastq.gz \
  /data/SRX19144488_SRR23195511_2.fastq.gz \
  -o /results

---

### Now that you know how to use a docker container, which approach was easier and which approach will be easier in the future?

Conda is simpler locally, Docker is better for reproducible repeated analyses

- **Conda was easier for a single local task**
    - fewer commands
    - simpler access to local files
    - can be slower
- **Docker is better for future workflows**
    - software and dependencies are already packaged
    - avoids dependency conflicts

---

### What would you say, which approach is more reproducible?

Docker is more reproducible
- fixes the software version and dependencies
- isolates the software from the local system
- reduces differences between computers
- makes it easier to reproduce the same analysis on different systems

---

### Compare the file to last weeks fastqc results, are they identical?
### Is the fastqc version identical?

- the FastQC results are identical
- read count, read length, GC content, duplication, and PASS/WARN/FAIL results match
- the FastQC version is also identical: 0.12.1

---

## Dockerfiles

We now used Docker containers and images directly to boost our research. 

Let's create our own toy Dockerfile including the "cowsay" tool (https://en.wikipedia.org/wiki/Cowsay)

Hints:
1. Docker is Linux, so you need to know the apt-get command to install "cowsay"

In [13]:
# open the file "my_dockerfile" in a text editor
!code my_dockerfile

---

### Explain the RUN and ENV lines you added to the file

- `RUN apt-get update && apt-get install -y curl cowsay`
  - `RUN` executes a command while the Docker image is being built
  - `apt-get update` refreshes the package list
  - `apt-get install -y curl cowsay` installs `curl` and `cowsay`
  - `-y` automatically confirms the installation

- `ENV PATH="/usr/games:${PATH}"`
  - `ENV` sets an environment variable inside the container
  - `cowsay` is installed in `/usr/games`
  - adding `/usr/games` to `PATH` allows `cowsay` to be run as `cowsay "Hello"` instead of `/usr/games/cowsay "Hello"`

In [ ]:
# build the docker image
!docker build -f my_dockerfile -t cowsay-toy .

In [ ]:
# make sure that the image has been built
!docker images

In [23]:
# run the docker file 
!docker run --rm cowsay-toy cowsay "°❀⋆.ೃ࿔*:･°❀⋆.ೃ࿔*"

 __
< °❀⋆.ೃ࿔*:･°❀⋆.ೃ࿔*  >
 --
        \   ^__^
         \  (oo)\_______
            (__)\       )\/\
                ||----w |
                ||     ||


---

## Let's do some bioinformatics with the docker file and create a new docker file that holds the salmon tool used in rnaseq

To do so, use "curl" in your new dockerfile to get salmon from https://github.com/COMBINE-lab/salmon/releases/download/v1.5.2/salmon-1.5.2_linux_x86_64.tar.gz

In [ ]:
# use the file "salmon_docker" in this directory to build a new docker image
!cat salmon_docker

In [ ]:
# build the image
!docker build \
  --platform linux/amd64 \
  -f salmon_docker \
  -t salmon-image .

In [40]:
# run the docker image to give out the version of salmon
!docker run --rm \
  --platform linux/amd64 \
  salmon-image \
  salmon --version

salmon 1.5.2


---

## Do you think bioinformaticians have to create a docker image every time they want to run a tool?

Find the salmon docker image online and run it on your computer.

What is https://biocontainers.pro/ ?

- no, they do not have to create a Docker image for every tool; many bioinformatics tools already have published container images
- Salmon 1.5.2 is available as `quay.io/biocontainers/salmon:1.5.2--h84f40af_0`
- BioContainers is a community-driven project that provides and maintains standardized container images for scientific software

---

In [36]:
!docker pull quay.io/biocontainers/salmon:1.5.2--h84f40af_0

1.5.2--h84f40af_0: Pulling from biocontainers/salmon

6a04cedd: Pulling fs layer 
59c44bf5: Pulling fs layer 
Digest: sha256:4ae09a47788f08317bd2f758ac4c8804c9e87d88caf500c449e18ac4794d0332
Status: Downloaded newer image for quay.io/biocontainers/salmon:1.5.2--h84f40af_0
quay.io/biocontainers/salmon:1.5.2--h84f40af_0


https://quay.io/repository/biocontainers/salmon?tab=info

tag `1.5.2--h84f40af_0`

In [37]:
!docker run --rm \
  --platform linux/amd64 \
  quay.io/biocontainers/salmon:1.5.2--h84f40af_0 \
  salmon --version

salmon 1.5.2


---

## Are there other ways to create Docker (or Apptainer) images?

What is https://seqera.io/containers/ ?

- Seqera Containers is a free service that automatically creates container images from Conda or PyPI packages
- users can select software packages, versions, architecture, and container format without writing a Dockerfile manually
- the generated images are hosted at `community.wave.seqera.io` and can be used with Docker or Singularity/Apptainer
- useful for quickly creating reproducible bioinformatics environments and integrating them into Nextflow workflows

---